# Part 3: PageRank on Spark
## CSL7110 - Machine Learning with Big Data | Assignment 4
**Name:** Aniket Srivastava | **Roll No:** M25DE1051

Implements iterative PageRank using PySpark RDDs.
- beta = 0.8, iterations = 40
- Tested on small graph (n=53) then full graph (n=1000)

In [ ]:
from pyspark import SparkContext, SparkConf
import os

conf = SparkConf().setAppName("PageRank_A4").setMaster("local[*]")
sc = SparkContext(conf=conf)
sc.setLogLevel("ERROR")
print("SparkContext initialized")

SparkContext initialized


## Load Edges and Build Adjacency List

In [ ]:
# Load edges, deduplicate, build adjacency list
def load_edges(sc, filepath):
    """Load edge list file, deduplicate, return RDD of (src, dst)."""
    raw = sc.textFile(filepath)
    edges = raw.map(lambda line: tuple(map(int, line.strip().split()))).distinct()
    return edges

def build_adjacency(edges_rdd):
    """Build (src, [dst1, dst2, ...]) adjacency list RDD."""
    return edges_rdd.groupByKey().mapValues(lambda nbrs: list(set(nbrs)))

print("Functions defined")

Functions defined


## Iterative PageRank Function

In [ ]:
# Iterative PageRank: r = (1-beta)/n * A + beta * M * r
def pagerank(sc, adj_rdd, n, beta=0.8, iterations=40):
    teleport = (1.0 - beta) / n
    ranks = adj_rdd.mapValues(lambda _: 1.0 / n)
    for i in range(iterations):
        contribs = adj_rdd.join(ranks).flatMap(
            lambda x: [(dst, x[1][1] / len(x[1][0])) for dst in x[1][0]]
        )
        ranks = contribs.reduceByKey(lambda a, b: a + b) \
                        .mapValues(lambda s: teleport + beta * s)
    return ranks

print("pagerank function defined")

pagerank function defined


## Run on Small Graph (n=53)

In [ ]:
SMALL_PATH = '../data/Q3/small.txt'

edges_small = load_edges(sc, SMALL_PATH)
adj_small = build_adjacency(edges_small)
n_small = 53

print(f"Small graph edges : {edges_small.count()}")
print(f"Small graph nodes : {adj_small.count()}")

ranks_small = pagerank(sc, adj_small, n=n_small, beta=0.8, iterations=40)
ranked_small = sorted(ranks_small.collect(), key=lambda x: -x[1])

print("\nTop 5 nodes (highest PageRank):")
for i, (node, score) in enumerate(ranked_small[:5], 1):
    print(f"  #{i}  Node {node:4d}  score = {score:.6f}")

print("\nBottom 5 nodes (lowest PageRank):")
for i, (node, score) in enumerate(ranked_small[-5:], 1):
    print(f"  #{i}  Node {node:4d}  score = {score:.6f}")

print(f"\nTop score: {ranked_small[0][1]:.6f}")

Small graph edges : 208
Small graph nodes : 53

Top 5 nodes (highest PageRank):
  #1  Node    5  score = 0.067613
  #2  Node    0  score = 0.066646
  #3  Node   20  score = 0.056184
  #4  Node   10  score = 0.054724
  #5  Node   15  score = 0.054326

Bottom 5 nodes (lowest PageRank):
  #1  Node   24  score = 0.006140
  #2  Node   14  score = 0.005952
  #3  Node   39  score = 0.005363
  #4  Node   52  score = 0.005316
  #5  Node    9  score = 0.004621

Top score: 0.067613


## Run on Full Graph (n=1000)

In [ ]:
WHOLE_PATH = '../data/Q3/whole.txt'

edges_whole = load_edges(sc, WHOLE_PATH)
adj_whole = build_adjacency(edges_whole)
n_whole = 1000

print(f"Whole graph edges : {edges_whole.count()}")
print(f"Whole graph nodes : {adj_whole.count()}")

ranks_whole = pagerank(sc, adj_whole, n=n_whole, beta=0.8, iterations=40)
ranked_whole = sorted(ranks_whole.collect(), key=lambda x: -x[1])

print("\nTop 5 node IDs (highest PageRank):")
for node, score in ranked_whole[:5]:
    print(f"  Node {node:4d}  score = {score:.6f}")

print("\nBottom 5 node IDs (lowest PageRank):")
for node, score in ranked_whole[-5:]:
    print(f"  Node {node:4d}  score = {score:.6f}")

Whole graph edges : 8390
Whole graph nodes : 1000

Top 5 node IDs (highest PageRank):
  Node  487  score = 0.002355
  Node  433  score = 0.001849
  Node  713  score = 0.001838
  Node  738  score = 0.001763
  Node  997  score = 0.001749

Bottom 5 node IDs (lowest PageRank):
  Node  272  score = 0.000405
  Node  403  score = 0.000402
  Node   96  score = 0.000376
  Node   21  score = 0.000367
  Node   13  score = 0.000242


In [ ]:
sc.stop()
print("SparkContext stopped")

SparkContext stopped
